In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os, sys
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from matplotlib.ticker import AutoMinorLocator
from IPython.display import HTML
import matplotlib.colors as mcolors
from matplotlib.animation import FFMpegWriter, PillowWriter
sys.path.insert(0, os.path.abspath('../code'))
from axion_solver import Params
from greens_solver import StepParams, E_tilde

plt.style.use('mine.mplstyle')
plt.rcParams["axes.axisbelow"] = False
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amssymb}",
    "animation.html": "html5",
    "animation.embed_limit": 256,
})
plt.rcParams['savefig.dpi'] = 'figure'
p = Params()
sp = StepParams()
L = p.L_region
Lb = p.L_box
ma = p.m_a
eps = 0.015
edges = [0.0, L, 2*L, 3*L, 4*L, 5*L]

In [ ]:
# Figure 1: the time-domain (ODE) solver against the frequency-domain Green's
# function, same piecewise-constant profile, with their difference underneath.
# The driving axion field rides in its own panel: a/a0 is O(1) while E_y is
# O(10), so it gets its own axis rather than an invented scale factor.
gr = np.load('../data/greens_run.npz')
x_gr = gr['x']
t_gr = gr['t']
E_gr = gr['E'] / p.gB0a0
E_td = gr['E_td'] / p.gB0a0
dE_gr = E_gr - E_td

step_gr = 2                      # every 2nd snapshot -> 301 frames, 12 s at 25 fps
frames_gr = np.arange(0, len(t_gr), step_gr)

xf_gr = np.linspace(x_gr[0], x_gr[-1], 4000)
wp_gr = sp.omega_p(xf_gr) / ma
ymax_gr = 1.08 * max(np.abs(E_gr).max(), np.abs(E_td).max())
dmax_gr = 1.15 * np.abs(dE_gr).max()
cs_solvers = ['tab:blue', 'tab:red']
c_axion = 'tab:green'

fig1 = plt.figure(figsize=(7.9, 7.9 / 1.61803398875 * 1.40), dpi=110)
gs1 = GridSpec(32, 40, figure=fig1)
ax_wp = fig1.add_subplot(gs1[0:5, 1:-1])
ax_ax = fig1.add_subplot(gs1[6:11, 1:-1])
ax_Ey = fig1.add_subplot(gs1[12:25, 1:-1])
ax_dE = fig1.add_subplot(gs1[26:32, 1:-1])

ax_wp.set_xlim(x_gr[0], x_gr[-1])
ax_wp.set_ylim(0.70, 1.34)
ax_wp.set_ylabel(r'$\omega_p/m_a$', labelpad=2)
ax_wp.set_xticklabels([])
ax_wp.axvspan(edges[2], edges[3], color='0.88')
ax_wp.axhline(1.0, lw=0.5, ls='--', c='0.45')
ax_wp.plot(xf_gr, wp_gr, lw=0.8, c='k')

ax_ax.set_xlim(x_gr[0], x_gr[-1])
ax_ax.set_ylim(-1.18, 1.18)
ax_ax.set_ylabel(r'$a/a_0$', labelpad=2)
ax_ax.set_xticklabels([])
ax_ax.axvspan(edges[2], edges[3], color='0.88')
ax_ax.axhline(0.0, lw=0.4, c='0.8')

ax_Ey.set_xlim(x_gr[0], x_gr[-1])
ax_Ey.set_ylim(-ymax_gr, ymax_gr)
ax_Ey.set_ylabel(r'$E_y\,/\,(g_{a\gamma\gamma}B_0a_0)$', labelpad=2)
ax_Ey.set_xticklabels([])
ax_Ey.axvspan(edges[2], edges[3], color='0.88')
ax_Ey.axhline(0.0, lw=0.4, c='0.8')

ax_dE.set_xlim(x_gr[0], x_gr[-1])
ax_dE.set_ylim(-dmax_gr, dmax_gr)
ax_dE.set_xlabel(r'$x$')
ax_dE.set_ylabel(r'$\Delta E_y$', labelpad=2)
ax_dE.axvspan(edges[2], edges[3], color='0.88')
ax_dE.axhline(0.0, lw=0.4, c='0.8')

for xe in edges:
    ax_wp.axvline(xe, lw=0.4, ls=':', c='0.6')
    ax_ax.axvline(xe, lw=0.4, ls=':', c='0.6')
    ax_Ey.axvline(xe, lw=0.4, ls=':', c='0.6')
    ax_dE.axvline(xe, lw=0.4, ls=':', c='0.6')

l_a, = ax_ax.plot([], [], lw=0.8, c=c_axion)
l_aenv, = ax_ax.plot([], [], lw=0.5, c='tab:green')
l_td, = ax_Ey.plot([], [], lw=0.9, c=cs_solvers[0], label=r'time-domain solver')
l_gr, = ax_Ey.plot([], [], lw=0.9, ls='--', c=cs_solvers[1], label=r"Green's function")
l_pk, = ax_Ey.plot([], [], lw=0.5, ls=':', c=c_axion)
l_dE, = ax_dE.plot([], [], lw=0.6, c='0.35')
txt_gr = ax_Ey.text(0.985, 0.92, '', transform=ax_Ey.transAxes, ha='right',
                    va='top', fontsize=8, usetex=False)
ax_Ey.legend(loc='upper left')

x_aenv = np.concatenate([x_gr, x_gr[::-1]])

def update_gr(j):
    xi = x_gr - p.x0 - p.v_a * t_gr[j]
    gauss = np.exp(-0.5 * (xi / p.sigma_x) ** 2)
    a_now = gauss * np.cos(p.k_a * (x_gr - p.x0) - p.omega_a * t_gr[j])
    l_a.set_data(x_gr, a_now)
    l_aenv.set_data(x_aenv, np.concatenate([gauss, -gauss[::-1]]))
    xc = p.x0 + p.v_a * t_gr[j]
    l_pk.set_data([xc, xc], [-ymax_gr, ymax_gr])
    l_td.set_data(x_gr, E_td[j])
    l_gr.set_data(x_gr, E_gr[j])
    l_dE.set_data(x_gr, dE_gr[j])
    txt_gr.set_text('t = %6.1f' % t_gr[j])
    return l_a, l_aenv, l_pk, l_td, l_gr, l_dE, txt_gr

ani1 = FuncAnimation(fig1, update_gr, frames=frames_gr, interval=40, blit=False)
plt.close(fig1)
#ani1.save('../figures/nb_solver_vs_greens.mp4', writer=FFMpegWriter(fps=25, bitrate=3600), dpi=110)
HTML(ani1.to_html5_video())

In [ ]:

def color_shades(color, n, lightest_alpha=0.25):
    """Generate n shades from `color` to lighter, blended toward white.

    Parameters
    ----------
    color : str or RGB tuple
        Base color (e.g. "tab:blue").
    n : int
        Number of shades.
    lightest_alpha : float
        Fraction of base color in the lightest shade (0=white, 1=original).

    Returns
    -------
    list of hex color strings, from darkest to lightest.
    """
    rgb = np.array(mcolors.to_rgb(color))
    white = np.array([1.0, 1.0, 1.0])
    fracs = np.linspace(1.0, lightest_alpha, n)
    return [mcolors.to_hex(rgb * f + white * (1 - f)) for f in fracs]
# Figure 2: five single-frequency modes, Re[E~ e^{-i phi}], animated against the
# mode's own phase so all five rows stay locked together. One shared absolute
# y-scale across the rows, so the panel heights compare the actual response.
oms_modes = np.array([0.85, 0.95, 1.00, 1.05, 1.15])



cs_modes = [color_shades('tab:blue',2), color_shades('tab:purple',2), color_shades('tab:red',2), 
            color_shades('tab:orange',2),color_shades('goldenrod',2)]
x_modes = np.linspace(-3.0, 55.0, 1200)
Et_modes = E_tilde(x_modes, oms_modes + 1j*eps, p) / p.gB0a0
#ylim_modes = 1.06 * np.abs(Et_modes).max()
ylim_modes = 58

xf_modes = np.linspace(x_modes[0], x_modes[-1], 4000)
wp_modes = sp.omega_p(xf_modes) / ma
phases = np.linspace(0.0, 4*np.pi, 180, endpoint=False)

fig2 = plt.figure(figsize=(7.9, 7.9 / 1.61803398875 * 2.05), dpi=110)
gs2 = GridSpec(31, 40, figure=fig2)
ax2_wp = fig2.add_subplot(gs2[0:5, 1:-1])
ax2_a = fig2.add_subplot(gs2[5:10, 1:-1])
ax2_b = fig2.add_subplot(gs2[10:15, 1:-1])
ax2_c = fig2.add_subplot(gs2[15:20, 1:-1])
ax2_d = fig2.add_subplot(gs2[20:25, 1:-1])
ax2_e = fig2.add_subplot(gs2[25:30, 1:-1])

ax2_wp.set_xlim(x_modes[0], x_modes[-1])
ax2_wp.set_ylim(0.74, 1.30)
ax2_wp.set_ylabel(r'$\omega_p/m_a$', labelpad=2)
ax2_wp.set_xticklabels([])
ax2_wp.axvspan(edges[2], edges[3], color='tab:gray',zorder=2,alpha=0.15)
ax2_wp.plot(xf_modes, wp_modes, lw=0.9, c='k')
ax2_wp.axhline(oms_modes[0], lw=0.6, c=cs_modes[0][0],ls='--')
ax2_wp.axhline(oms_modes[1], lw=0.6, c=cs_modes[1][0],ls='--')
ax2_wp.axhline(oms_modes[2], lw=0.6, c=cs_modes[2][0],ls='--')
ax2_wp.axhline(oms_modes[3], lw=0.6, c=cs_modes[3][0],ls='--')
ax2_wp.axhline(oms_modes[4], lw=0.6, c=cs_modes[4][0],ls='--')

ax2_a.set_xlim(x_modes[0], x_modes[-1])
ax2_a.set_ylim(-ylim_modes, ylim_modes)
ax2_a.text(54,-48,r'$\omega/m_a=0.85$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax2_a.set_xticklabels([])
ax2_a.axvspan(edges[2], edges[3], color='tab:gray', zorder=2,alpha=0.15)
ax2_a.fill_between(x_modes, -np.abs(Et_modes[0]), np.abs(Et_modes[0]), color=cs_modes[0][1], lw=0)
l2_a, = ax2_a.plot([], [], lw=1.0, c=cs_modes[0][0])

ax2_b.set_xlim(x_modes[0], x_modes[-1])
ax2_b.set_ylim(-ylim_modes, ylim_modes)
ax2_b.text(54,-48,r'$\omega/m_a=0.95$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax2_b.set_xticklabels([])
ax2_b.axvspan(edges[2], edges[3],  color='tab:gray',zorder=2,alpha=0.15)
ax2_b.fill_between(x_modes, -np.abs(Et_modes[1]), np.abs(Et_modes[1]), color=cs_modes[1][1], lw=0)
l2_b, = ax2_b.plot([], [], lw=1.0, c=cs_modes[1][0])

ax2_c.set_xlim(x_modes[0], x_modes[-1])
ax2_c.set_ylim(-ylim_modes, ylim_modes)
ax2_c.text(54,-48,r'$\omega/m_a=1.0$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax2_c.set_xticklabels([])
ax2_c.axvspan(edges[2], edges[3], color='tab:gray',zorder=2,alpha=0.15)
ax2_c.fill_between(x_modes, -np.abs(Et_modes[2]), np.abs(Et_modes[2]), color=cs_modes[2][1], lw=0)
l2_c, = ax2_c.plot([], [], lw=1.0, c=cs_modes[2][0])

ax2_d.set_xlim(x_modes[0], x_modes[-1])
ax2_d.set_ylim(-ylim_modes, ylim_modes)
ax2_d.text(54,-48,r'$\omega/m_a=1.05$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax2_d.set_xticklabels([])
ax2_d.axvspan(edges[2], edges[3],  color='tab:gray',zorder=2,alpha=0.15)
ax2_d.fill_between(x_modes, -np.abs(Et_modes[3]), np.abs(Et_modes[3]), color=cs_modes[3][1], lw=0)
l2_d, = ax2_d.plot([], [], lw=1.0, c=cs_modes[3][0])

ax2_e.set_xlim(x_modes[0], x_modes[-1])
ax2_e.set_ylim(-ylim_modes, ylim_modes)
ax2_e.text(54,-48,r'$\omega/m_a=1.15$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax2_e.set_xlabel(r'$x$')
ax2_e.axvspan(edges[2], edges[3],  color='tab:gray',zorder=2,alpha=0.15)
ax2_e.fill_between(x_modes, -np.abs(Et_modes[4]), np.abs(Et_modes[4]), color=cs_modes[4][1], lw=0)
l2_e, = ax2_e.plot([], [], lw=1.0, c=cs_modes[4][0])

ax2_c.set_ylabel(r'$\mathrm{Re}\,[\,\tilde{E}_y(x,\omega)\,e^{-i\varphi}\,]\,/\,(g_{a\gamma\gamma}B_0a_0)$')
txt_ph = ax2_wp.text(0.985, 0.88, '', transform=ax2_wp.transAxes, ha='right',
                     va='top', fontsize=8, usetex=False)

def update_phase(k):
    l2_a.set_data(x_modes, (Et_modes[0]*np.exp(-1j*phases[k])).real)
    l2_b.set_data(x_modes, (Et_modes[1]*np.exp(-1j*phases[k])).real)
    l2_c.set_data(x_modes, (Et_modes[2]*np.exp(-1j*phases[k])).real)
    l2_d.set_data(x_modes, (Et_modes[3]*np.exp(-1j*phases[k])).real)
    l2_e.set_data(x_modes, (Et_modes[4]*np.exp(-1j*phases[k])).real)
    txt_ph.set_text(r'$\varphi\equiv \omega t$ = %4.2f' % (phases[k] % (2*np.pi)))
    return l2_a, l2_b, l2_c, l2_d, l2_e, txt_ph

ani2 = FuncAnimation(fig2, update_phase, frames=len(phases), interval=42, blit=False)
plt.close(fig2)
ani2.save('../figures/nb_modes_vs_phase.mp4',   writer=FFMpegWriter(fps=24, bitrate=3600), dpi=110)
HTML(ani2.to_html5_video())

In [ ]:
# Figure 3: the same five modes against real time. phi = omega t, so each row
# now runs at its own rate: they start aligned at t = 0 and drift apart. The
# axion packet envelope rides along on top -- here it is a genuine time, so
# x_c = x0 + v_a t is well defined. Reuses Et_modes / cs_modes from the cell above.
times = np.linspace(0.0, 150.0, 300, endpoint=False)   # packet reaches and crosses slab 3
env_amp = 0.92 * ylim_modes
x_env = np.concatenate([x_modes, x_modes[::-1]])

fig3 = plt.figure(figsize=(7.9, 7.9 / 1.61803398875 * 2.05), dpi=110)
gs3 = GridSpec(31, 40, figure=fig3)
ax3_wp = fig3.add_subplot(gs3[0:5, 1:-1])
ax3_a = fig3.add_subplot(gs3[5:10, 1:-1])
ax3_b = fig3.add_subplot(gs3[10:15, 1:-1])
ax3_c = fig3.add_subplot(gs3[15:20, 1:-1])
ax3_d = fig3.add_subplot(gs3[20:25, 1:-1])
ax3_e = fig3.add_subplot(gs3[25:30, 1:-1])

ax3_wp.set_xlim(x_modes[0], x_modes[-1])
ax3_wp.set_ylim(0.74, 1.30)
ax3_wp.set_ylabel(r'$\omega_p/m_a$', labelpad=2)
ax3_wp.set_xticklabels([])
ax3_wp.axvspan(edges[2], edges[3], color='tab:gray', zorder=2, alpha=0.15)
ax3_wp.plot(xf_modes, wp_modes, lw=0.9, c='k')
ax3_wp.axhline(oms_modes[0], lw=0.6, c=cs_modes[0][0], ls='--')
ax3_wp.axhline(oms_modes[1], lw=0.6, c=cs_modes[1][0], ls='--')
ax3_wp.axhline(oms_modes[2], lw=0.6, c=cs_modes[2][0], ls='--')
ax3_wp.axhline(oms_modes[3], lw=0.6, c=cs_modes[3][0], ls='--')
ax3_wp.axhline(oms_modes[4], lw=0.6, c=cs_modes[4][0], ls='--')

ax3_a.set_xlim(x_modes[0], x_modes[-1])
ax3_a.set_ylim(-ylim_modes, ylim_modes)
ax3_a.text(54, -48, r'$\omega/m_a=0.85$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax3_a.set_xticklabels([])
ax3_a.axvspan(edges[2], edges[3], color='tab:gray', zorder=2, alpha=0.15)
ax3_a.fill_between(x_modes, -np.abs(Et_modes[0]), np.abs(Et_modes[0]), color=cs_modes[0][1], lw=0)
l3_env_a, = ax3_a.plot([], [], lw=0.6, c='tab:green', ls='-')
l3_a, = ax3_a.plot([], [], lw=1.0, c=cs_modes[0][0])

ax3_b.set_xlim(x_modes[0], x_modes[-1])
ax3_b.set_ylim(-ylim_modes, ylim_modes)
ax3_b.text(54, -48, r'$\omega/m_a=0.95$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax3_b.set_xticklabels([])
ax3_b.axvspan(edges[2], edges[3], color='tab:gray', zorder=2, alpha=0.15)
ax3_b.fill_between(x_modes, -np.abs(Et_modes[1]), np.abs(Et_modes[1]), color=cs_modes[1][1], lw=0)
l3_env_b, = ax3_b.plot([], [], lw=0.6, c='tab:green', ls='-')
l3_b, = ax3_b.plot([], [], lw=1.0, c=cs_modes[1][0])

ax3_c.set_xlim(x_modes[0], x_modes[-1])
ax3_c.set_ylim(-ylim_modes, ylim_modes)
ax3_c.text(54, -48, r'$\omega/m_a=1.0$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax3_c.set_xticklabels([])
ax3_c.axvspan(edges[2], edges[3], color='tab:gray', zorder=2, alpha=0.15)
ax3_c.fill_between(x_modes, -np.abs(Et_modes[2]), np.abs(Et_modes[2]), color=cs_modes[2][1], lw=0)
l3_env_c, = ax3_c.plot([], [], lw=0.6, c='tab:green', ls='-')
l3_c, = ax3_c.plot([], [], lw=1.0, c=cs_modes[2][0])

ax3_d.set_xlim(x_modes[0], x_modes[-1])
ax3_d.set_ylim(-ylim_modes, ylim_modes)
ax3_d.text(54, -48, r'$\omega/m_a=1.05$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax3_d.set_xticklabels([])
ax3_d.axvspan(edges[2], edges[3], color='tab:gray', zorder=2, alpha=0.15)
ax3_d.fill_between(x_modes, -np.abs(Et_modes[3]), np.abs(Et_modes[3]), color=cs_modes[3][1], lw=0)
l3_env_d, = ax3_d.plot([], [], lw=0.6, c='tab:green', ls='-')
l3_d, = ax3_d.plot([], [], lw=1.0, c=cs_modes[3][0])

ax3_e.set_xlim(x_modes[0], x_modes[-1])
ax3_e.set_ylim(-ylim_modes, ylim_modes)
ax3_e.text(54, -48, r'$\omega/m_a=1.15$', fontsize=mpl.rcParams['legend.fontsize'], ha='right')
ax3_e.set_xlabel(r'$x$')
ax3_e.axvspan(edges[2], edges[3], color='tab:gray', zorder=2, alpha=0.15)
ax3_e.fill_between(x_modes, -np.abs(Et_modes[4]), np.abs(Et_modes[4]), color=cs_modes[4][1], lw=0)
l3_env_e, = ax3_e.plot([], [], lw=0.6, c='tab:green', ls='-')
l3_e, = ax3_e.plot([], [], lw=1.0, c=cs_modes[4][0])

ax3_c.set_ylabel(r'$\mathrm{Re}\,[\,\tilde{E}_y(x,\omega)\,e^{-i\omega t}\,]\,/\,(g_{a\gamma\gamma}B_0a_0)$')
txt_tm = ax3_wp.text(0.985, 0.88, '', transform=ax3_wp.transAxes, ha='right',
                     va='top', fontsize=8, usetex=False)

def update_time(k):
    l3_a.set_data(x_modes, (Et_modes[0]*np.exp(-1j*oms_modes[0]*times[k])).real)
    l3_b.set_data(x_modes, (Et_modes[1]*np.exp(-1j*oms_modes[1]*times[k])).real)
    l3_c.set_data(x_modes, (Et_modes[2]*np.exp(-1j*oms_modes[2]*times[k])).real)
    l3_d.set_data(x_modes, (Et_modes[3]*np.exp(-1j*oms_modes[3]*times[k])).real)
    l3_e.set_data(x_modes, (Et_modes[4]*np.exp(-1j*oms_modes[4]*times[k])).real)
    xc = p.x0 + p.v_a * times[k]
    env = env_amp * np.exp(-0.5*((x_modes - xc)/p.sigma_x)**2)
    y_env = np.concatenate([env, -env[::-1]])
    l3_env_a.set_data(x_env, y_env)
    l3_env_b.set_data(x_env, y_env)
    l3_env_c.set_data(x_env, y_env)
    l3_env_d.set_data(x_env, y_env)
    l3_env_e.set_data(x_env, y_env)
    txt_tm.set_text(r'$t$ = %5.1f' % times[k])
    return (l3_a, l3_b, l3_c, l3_d, l3_e, l3_env_a, l3_env_b,
            l3_env_c, l3_env_d, l3_env_e, txt_tm)

ani3 = FuncAnimation(fig3, update_time, frames=len(times), interval=42, blit=False)
plt.close(fig3)
ani3.save('../figures/nb_modes_vs_time.mp4', writer=FFMpegWriter(fps=24, bitrate=3600), dpi=110)
HTML(ani3.to_html5_video())